# 01 · Model architecture

**Workshop #2 — Team 4, sentiment analysis.** Architecture of the final model
(`checkpoints/model_best.pt`): what it is, how many parameters it has, and a forward
pass on a sample batch with the output shapes checked, as the brief requires.

The model is a pretrained **RoBERTa-base** encoder with a small classification head:

```
input_ids (batch, ≤256) ──► RoBERTa encoder: 12 layers · hidden 768 · 12 heads · GELU
                                 │ last_hidden_state[:, 0]   (<s> token, the [CLS] equivalent)
                                 ▼
                          Dropout(0.1) ──► Linear(768 → 3) ──► logits (batch, 3)
```

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"  # no progress bars in the rendered notebook
warnings.filterwarnings("ignore", category=UserWarning)

# Notebooks run with notebooks/ as the working directory; make `src` importable.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import plots
from src.config import LABELS, RUNS_DIR

plots.apply_theme()
pd.set_option("display.precision", 4)

import torch
from src.model import count_parameters, load_checkpoint

/home/juliancelis/Documents/Universidad/noveno_semestre/ML/segundo_taller/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the checkpoint (RoBERTa-base, final model)

The checkpoint stores the weights together with the full training configuration, so the
model is rebuilt here without `train.py`.

In [2]:
device = torch.device("cpu")  # a forward pass on a few reviews needs no GPU
model, tokenizer, config, checkpoint = load_checkpoint(ROOT / "checkpoints" / "model_best.pt", device)
print(f"model      : {config.model_name}")
print(f"best epoch : {checkpoint['epoch']} ({checkpoint['phase']} phase)")
print(f"{config.model_name} validation macro-F1 at save time: {checkpoint['val_metrics']['macro_f1']:.4f}")

model      : roberta-base
best epoch : 4 (finetune phase)
roberta-base validation macro-F1 at save time: 0.7350


## 2. Parameters (RoBERTa-base)

Counted by block. The encoder's pooler is part of the pretrained checkpoint but our head
reads the `<s>` token from `last_hidden_state` directly, so the pooler's weights exist
but receive no gradient signal from the task.

In [3]:
def n_params(module):
    return sum(p.numel() for p in module.parameters())

encoder = model.encoder
blocks = {
    "Embeddings (token + position + layer norm)": n_params(encoder.embeddings),
    "Transformer layers (12 × self-attention + FFN)": n_params(encoder.encoder),
    "Pooler (unused by the head)": n_params(encoder.pooler) if encoder.pooler is not None else 0,
    "Classification head (Linear 768 → 3)": n_params(model.classifier),
}
table = pd.DataFrame({"parameters": blocks})
table.loc["Total"] = table["parameters"].sum()
table["share"] = (table["parameters"] / table.loc["Total", "parameters"]).map("{:.2%}".format)
table["parameters"] = table["parameters"].map("{:,}".format)
table

,parameters,share
Embeddings (token + position + layer norm),"39,000,576",31.29%
Transformer layers (12 × self-attention + FFN),"85,054,464",68.24%
Pooler (unused by the head),"590,592",0.47%
Classification head (Linear 768 → 3),"2,307",0.00%
Total,"124,647,939",100.00%


In [4]:
counts = count_parameters(model)
model.freeze_encoder(True)
frozen_trainable = count_parameters(model)["trainable"]
model.freeze_encoder(False)
print(f"phase 1 (encoder frozen): {frozen_trainable:>12,} trainable parameters")
print(f"phase 2 (full fine-tune): {counts['trainable']:>12,} trainable parameters")

phase 1 (encoder frozen):        2,307 trainable parameters
phase 2 (full fine-tune):  124,647,939 trainable parameters


## 3. Layer dimensions (RoBERTa-base)

In [5]:
cfg = encoder.config
pd.DataFrame({"value": {
    "Transformer layers": cfg.num_hidden_layers,
    "Hidden size": cfg.hidden_size,
    "Attention heads": cfg.num_attention_heads,
    "Feed-forward size": cfg.intermediate_size,
    "Activation": cfg.hidden_act,
    "Vocabulary size": cfg.vocab_size,
    "Max position embeddings": cfg.max_position_embeddings,
    "Encoder dropout": cfg.hidden_dropout_prob,
    "Head dropout": config.dropout,
    "Input length used": config.max_len,
    "Output classes": len(LABELS),
}})

,value
Transformer layers,12
Hidden size,768
Attention heads,12
Feed-forward size,3072
Activation,gelu
Vocabulary size,50265
Max position embeddings,514
Encoder dropout,0.1
Head dropout,0.1
Input length used,256


## 4. Forward pass on a sample batch

Three reviews written for this check (not from the corpus), tokenised exactly as in
training: head+tail truncation to 256 tokens and dynamic padding.

In [6]:
from src.data import encode_head_tail, make_collate

sample = [
    "Absolutely magical day, the kids loved every ride and the staff were wonderful.",
    "It was okay. Some rides were fun but the queues were long and food was overpriced.",
    "Worst theme park experience ever. Rude staff, broken rides, and filthy toilets.",
]
ids = encode_head_tail(sample, tokenizer, config.max_len, config.head_tokens)
batch = make_collate(tokenizer.pad_token_id)([(seq, 0) for seq in ids])

with torch.no_grad():
    logits = model(batch["input_ids"], batch["attention_mask"])
probabilities = torch.softmax(logits, dim=-1)

print("input_ids      :", tuple(batch["input_ids"].shape), " (batch, seq_len)")
print("attention_mask :", tuple(batch["attention_mask"].shape))
print("logits         :", tuple(logits.shape), " (batch, classes)")
assert logits.shape == (len(sample), len(LABELS))
assert torch.allclose(probabilities.sum(dim=-1), torch.ones(len(sample)))

pd.DataFrame(probabilities.numpy(), columns=LABELS,
             index=[s[:60] + "…" for s in sample]).style.format("{:.3f}")

input_ids      : (3, 21)  (batch, seq_len)
attention_mask : (3, 21)
logits         : (3, 3)  (batch, classes)


,negative,neutral,positive
"Absolutely magical day, the kids loved every ride and the st…",0.000,0.000,1.000
It was okay. Some rides were fun but the queues were long an…,0.054,0.927,0.019
"Worst theme park experience ever. Rude staff, broken rides, …",0.999,0.001,0.000


## 5. Training configuration

Every hyperparameter of the run that produced the checkpoint, read from the checkpoint
itself.

In [7]:
pd.DataFrame({"value": {k: str(v) for k, v in config.to_dict().items()}})

,value
model_name,roberta-base
max_len,256
head_tokens,64
batch_size,32
eval_batch_size,64
frozen_epochs,1
head_lr,0.001
finetune_epochs,3
lr,2e-05
weight_decay,0.01


## Findings

- **124.6 M parameters.** 68 % sit in the 12 transformer layers and 31 % in the embeddings
  (a 50,265-token vocabulary × 768 dimensions). The task-specific head is only **2,307**
  parameters (0.002 %).
- **Phase 1 trains 0.002 % of the network**: with the encoder frozen only the head learns,
  which tests how much sentiment the pretrained representation already encodes before any
  fine-tuning (notebook 02 answers: almost as much as the best Workshop #1 model).
- **The forward pass is verified**: a batch of 3 reviews gives `logits` of shape `(3, 3)`
  and probabilities that sum to 1. Batches are padded only to their longest review
  (21 tokens here), not to 256.
- The pooler's 590 K parameters are loaded but unused, since the head reads the `<s>`
  token directly.